# Fake News Detection using NLP

## Research Paper Project

In this project, we use Natural Language Processing (NLP) techniques to detect fake news.
We combine TF-IDF vectorization with a PassiveAggressiveClassifier to classify news articles as REAL or FAKE.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import re
import itertools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
# Load True and Fake news datasets
true_df = pd.read_csv('True.csv')
true_df['label'] = 0  # REAL
fake_df = pd.read_csv('Fake.csv')
fake_df['label'] = 1  # FAKE

# Combine datasets
df = pd.concat([true_df, fake_df], ignore_index=True)
print('Dataset Shape:', df.shape)
print(df.head())

## 3. Data Preprocessing

In [ ]:
# Use only text and label columns
df = df[['text', 'label']].dropna()

# Features and labels
X = df['text']
y = df['label']

print('Total samples:', len(df))
print(df['label'].value_counts())

## 4. Split Train and Test Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## 5. TF-IDF Vectorization

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)
print('TF-IDF Train Shape:', tfidf_train.shape)
print('TF-IDF Test Shape:', tfidf_test.shape)

## 6. Train Model - PassiveAggressiveClassifier

In [ ]:
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(tfidf_train, y_train)

## 7. Evaluate Model

In [ ]:
# Predictions
y_pred = pac.predict(tfidf_test)

# Accuracy
score = accuracy_score(y_test, y_pred)
print('Accuracy:', round(score*100, 2), '%')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print('Confusion Matrix:\n', cm)

# Classification Report
print('Classification Report:\n', classification_report(y_test, y_pred))

## 8. Confusion Matrix Visualization

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
plt.title('Confusion Matrix - Fake News Detection using NLP')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 9. Demo Prediction

In [ ]:
def predict_news(text):
    tfidf_input = tfidf_vectorizer.transform([text])
    pred = pac.predict(tfidf_input)[0]
    prob = pac.predict_proba(tfidf_input).max()
    label = 'FAKE' if pred == 1 else 'REAL'
    return label, prob

# Test samples
sample1 = 'US President Donald Trump inaugurated in 2025.'
sample2 = 'Aliens landed in Uttarakhand yesterday.'

print('Sample 1:', predict_news(sample1))
print('Sample 2:', predict_news(sample2))